A while back, MedRxiv rejected our submission out of concern for potentially identifying information.
Need to submit a revised manuscript and source data anonymizing the offending data.

Relevant medRxiv message:
- Precise ages (e.g. "Age at diagnosis" in multiple tabs of "Supplemental Tables.xlsx") should be removed or 
replaced with an age range, e.g. in their 20’s (5-year range minimum) or with an age range in quartiles based on 
the age range of your study. Age ranges cannot overlap. Example of non-overlapping age ranges: 16-20, 21-25, 
26-30, etc. 

Sample and patient IDs are ok.

## Load data and packages

In [ ]:
import pandas as pd
import numpy as np
import warnings
import inspect

# import data_imports.py
import sys
sys.path.append('../src')
from data_imports import *

In [ ]:
patients = import_patients()
biosamples = import_biosamples()
# ... 

## Anonymize age at diagnosis

In [ ]:
def days_to_year_range(d,n):
    if np.isnan(d):
        return 'NA'
    age_bin = int(d // (365.25*n))
    age_bin = f'{n*age_bin}-{n*(age_bin+1)}'
    return age_bin
    
def anonymize_age(df,n=5):
    '''
    Inputs 
        df: pandas dataframe containing the  'age_at_diagnosis' column with integer values.
            Unit for age_at_diagnosis is expected to be days.
        n: year interval for anonymized age ranges. Default: 5 year intervals.
    Returns: pd dataframe with age_at_diagnosis replaced with age_range_at_diagnosis
    '''
    df["age_at_diagnosis"] = df["age_at_diagnosis"].map(lambda x: days_to_year_range(x,n))
    df = df.rename(columns={"age_at_diagnosis": "age_at_diagnosis_range"})
    return df

In [ ]:
def test_days_to_year_range():
    res1=days_to_year_range(np.nan,5)
    assert  res1 == 'NA', res1
    res2 = days_to_year_range(4495,5)
    assert res2 == '10-15',res2
    return f'pass: {inspect.currentframe().f_code.co_name}'
def test_anonymize_age():
    df = pd.DataFrame({
        "UID": ["A123", "B456"],
        "age_at_diagnosis": [810, np.nan],
        "something else": ["foo", "bar"]
    })
    df2 = pd.DataFrame({
        "UID": ["A123", "B456"],
        "age_at_diagnosis_range": ["0-5", "NA"],
        "something else": ["foo", "bar"]
    })
    res = anonymize_age(df,5)
    assert res.equals(df2), res
    return f'pass: {inspect.currentframe().f_code.co_name}'


In [ ]:
test_days_to_year_range()
test_anonymize_age()

## Anonymize patient and biosample IDs

In [ ]:
import uuid
import base64

def compact_uuid():
    u = uuid.uuid4()
    # URL-safe base64 encoding of the UUID bytes
    return base64.urlsafe_b64encode(u.bytes).rstrip(b'=').decode('ascii')

def assign_uuids(input_set,prefix=''):
    """Map each element in the input set to a unique compact UUID."""
    return {item: compact_uuid() for item in input_set}